# Tugas Sistem Temu Balik Informasi

Membandingkan performa dari metode ranked retrieval pada dataset News.csv. Konten yang diambil hanya pada kolom `content`.

Metode:
1. TF IDF (Cosine Similarity) (Vector Space)
    a. TF
    b. TF IDF Word2Vec
2. Query-Likelihood Retrieval Model (Probabilistic Approach)
    a. No Smoothing
    b. Add-One (Laplace)
    c. Linear Interpolation

In [1]:
# Menghubungkan ke Google Drive (jika menggunakan Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    path_to_data = '/content/drive/MyDrive/STBI_Tugas1/News.csv' # Sesuaikan path ini jika Anda menyimpan di folder lain
except:
    print("Not running in Google Colab. Using local path.")
    path_to_data = 'News.csv'

Not running in Google Colab. Using local path.


In [2]:
%pip install -q gensim

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
from collections import Counter
import math
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/vickymahfudy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vickymahfudy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/vickymahfudy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## 1. Load & Preprocessing Data

In [4]:
# Load dataset
try:
    df = pd.read_csv(path_to_data)
    # Gunakan hanya kolom 'content'
    if 'content' not in df.columns:
        print("Kolom 'content' tidak ditemukan. Pastikan dataset benar.")
    else:
        # Bersihkan missing values
        documents = df['content'].fillna('').tolist()
        print(f"Total documents: {len(documents)}")

        # UNTUK PERCOBAAN CEPAT (Bisa dihapus jika ingin menggunakan semua data)
        # Karena dataset bisa sangat besar, kita gunakan 5000 dokumen pertama untuk percobaan
        # documents = documents[:5000]
except Exception as e:
    print(f"Gagal memuat dataset: {e}")
    documents = []

Total documents: 14343


In [5]:
stop_words = set(stopwords.words('indonesian'))

def preprocess(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    return tokens

print("Preprocessing documents...")
if len(documents) > 0:
    processed_docs = [preprocess(doc) for doc in documents]
    print("Preprocessing selesai.")

Preprocessing documents...


Preprocessing selesai.


## 2. Model 1: Vector Space Model

### a. TF (Cosine Similarity)

In [6]:
# a. TF (menggunakan CountVectorizer dan Cosine Similarity)
if len(documents) > 0:
    vectorizer_tf = CountVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)
    X_tf = vectorizer_tf.fit_transform(processed_docs)

    def search_tf(query, top_n=5):
        q_processed = preprocess(query)
        q_vec = vectorizer_tf.transform([q_processed])
        similarities = cosine_similarity(q_vec, X_tf).flatten()
        top_indices = similarities.argsort()[-top_n:][::-1]
        return [(i, similarities[i]) for i in top_indices if similarities[i] > 0]

### b. TF-IDF (Cosine Similarity)

In [7]:
# b. TF-IDF (menggunakan TfidfVectorizer dan Cosine Similarity)
if len(documents) > 0:
    vectorizer_tfidf_only = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)
    X_tfidf_only = vectorizer_tfidf_only.fit_transform(processed_docs)

    def search_tfidf(query, top_n=5):
        q_processed = preprocess(query)
        q_vec = vectorizer_tfidf_only.transform([q_processed])
        similarities = cosine_similarity(q_vec, X_tfidf_only).flatten()
        top_indices = similarities.argsort()[-top_n:][::-1]
        return [(i, similarities[i]) for i in top_indices if similarities[i] > 0]


### c. Word2Vec

In [8]:
if len(documents) > 0:
    # Latih Word2Vec dari dokumen kita sendiri agar vocabulary sesuai
    print("Melatih model Word2Vec...")
    w2v_model = Word2Vec(sentences=processed_docs, vector_size=100, window=5, min_count=1, workers=4)

    print("Menghitung TF-IDF weights...")
    vectorizer_tfidf = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)
    X_tfidf = vectorizer_tfidf.fit_transform(processed_docs)
    tfidf_features = vectorizer_tfidf.get_feature_names_out()
    tfidf_weights = dict(zip(tfidf_features, vectorizer_tfidf.idf_))

    def get_doc_vector(doc_tokens):
        vec = np.zeros(100)
        weight_sum = 0
        for token in doc_tokens:
            if token in w2v_model.wv and token in tfidf_weights:
                tf = doc_tokens.count(token) # Term frequency sederhana
                idf = tfidf_weights[token]
                tf_idf = tf * idf
                vec += w2v_model.wv[token] * tf_idf
                weight_sum += tf_idf
        if weight_sum > 0:
            vec /= weight_sum
        return vec

    print("Membangun representasi vektor dokumen dengan TF-IDF + Word2Vec...")
    doc_vectors = np.array([get_doc_vector(doc) for doc in processed_docs])

    def search_tfidf_w2v(query, top_n=5):
        q_processed = preprocess(query)
        q_vec = get_doc_vector(q_processed).reshape(1, -1)
        if np.sum(q_vec) == 0:
            return []
        similarities = cosine_similarity(q_vec, doc_vectors).flatten()
        top_indices = similarities.argsort()[-top_n:][::-1]
        return [(i, similarities[i]) for i in top_indices if similarities[i] > 0]

Melatih model Word2Vec...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Menghitung TF-IDF weights...


Membangun representasi vektor dokumen dengan TF-IDF + Word2Vec...


## 3. Model 2: Query-Likelihood Retrieval Model (Probabilistic Approach)

In [9]:
if len(documents) > 0:
    # Bangun index dan statistik koleksi untuk model probabilitas
    print("Membangun statistik koleksi untuk QL Model...")
    doc_lengths = [len(doc) for doc in processed_docs]
    collection_length = sum(doc_lengths)

    collection_counts = Counter()
    for doc in processed_docs:
        collection_counts.update(doc)

    vocab_size = len(collection_counts)

    # Term frequencies per document (untuk akses cepat)
    doc_tfs = [Counter(doc) for doc in processed_docs]

Membangun statistik koleksi untuk QL Model...


### a. No Smoothing

In [10]:
if len(documents) > 0:
    def search_ql_no_smoothing(query, top_n=5):
        q_processed = preprocess(query)
        scores = []
        for i, doc_tf in enumerate(doc_tfs):
            if doc_lengths[i] == 0:
                scores.append((i, float('-inf')))
                continue
            score = 0
            valid = True
            for q_term in q_processed:
                tf = doc_tf.get(q_term, 0)
                if tf == 0:
                    # Jika term tidak ada di dokumen, probabilitas jadi 0 (log(0) = -inf)
                    valid = False
                    break
                prob = tf / doc_lengths[i]
                score += math.log(prob)
            if valid:
                scores.append((i, score))
            else:
                scores.append((i, float('-inf')))

        # Sort berdasarkan score tertinggi
        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return [s for s in scores[:top_n] if s[1] != float('-inf')]

### b. Add-One (Laplace) Smoothing

In [11]:
if len(documents) > 0:
    def search_ql_laplace(query, top_n=5):
        q_processed = preprocess(query)
        scores = []
        for i, doc_tf in enumerate(doc_tfs):
            score = 0
            doc_len = doc_lengths[i]
            for q_term in q_processed:
                tf = doc_tf.get(q_term, 0)
                # Probabilitas dengan smoothing Laplace
                prob = (tf + 1) / (doc_len + vocab_size)
                score += math.log(prob)
            scores.append((i, score))

        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return scores[:top_n]

### c. Linear Interpolation (Jelinek-Mercer)

In [12]:
if len(documents) > 0:
    def search_ql_jelinek_mercer(query, lambda_param=0.5, top_n=5):
        q_processed = preprocess(query)
        scores = []
        for i, doc_tf in enumerate(doc_tfs):
            score = 0
            doc_len = doc_lengths[i]
            for q_term in q_processed:
                tf = doc_tf.get(q_term, 0)
                # P(w|d)
                p_ml_doc = tf / doc_len if doc_len > 0 else 0
                # P(w|C)
                p_ml_coll = collection_counts.get(q_term, 0) / collection_length if collection_length > 0 else 0

                # Linear Interpolation
                prob = (lambda_param * p_ml_doc) + ((1 - lambda_param) * p_ml_coll)

                if prob > 0:
                    score += math.log(prob)
                else:
                    # Jika term tidak ada di doc maupun di koleksi
                    score += float('-inf')
            scores.append((i, score))

        scores = sorted(scores, key=lambda x: x[1], reverse=True)
        return [s for s in scores[:top_n] if s[1] != float('-inf')]

## 4. Testing dan Perbandingan

In [13]:
import time

if len(documents) > 0:
    # 5 contoh query yang akan diuji
    queries = [
        "ppkm darurat warga negara asing dilarang masuk",
        "olimpiade tokyo bulu tangkis indonesia medali",
        "covid 19 isolasi mandiri protokol kesehatan",
        "bantuan sembako warga terdampak pandemi",
        "polisi korban tewas pembunuhan"
    ]

    models = {
        "TF": search_tf,
        "TF-IDF": search_tfidf,
        "Word2Vec (TF-IDF weighted)": search_tfidf_w2v,
        "QL (No Smoothing)": search_ql_no_smoothing,
        "QL (Laplace)": search_ql_laplace,
        "QL (Linear Interpolation)": lambda q, top_n=10: search_ql_jelinek_mercer(q, lambda_param=0.5, top_n=top_n),
    }

    result_rows = []
    timing_rows = []

    for query_test in queries:
        for model_name, search_fn in models.items():
            start = time.perf_counter()
            res = search_fn(query_test, top_n=10)
            elapsed = time.perf_counter() - start

            timing_rows.append({
                "query": query_test,
                "model": model_name,
                "time_sec": elapsed,
                "n_results": len(res),
            })

            q_terms = set(preprocess(query_test))
            for rank, (doc_id, score) in enumerate(res, start=1):
                doc_terms = set(processed_docs[doc_id])
                overlap = len(q_terms & doc_terms)
                result_rows.append({
                    "query": query_test,
                    "model": model_name,
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": score,
                    "term_overlap": overlap,          # proksi otomatis relevansi (jumlah query term yg muncul di dok)
                    "term_overlap_ratio": overlap / len(q_terms) if q_terms else 0,
                    "content_snippet": documents[doc_id][:200],
                    "relevan_manual": "",              # kolom kosong, isi manual (Y/N) saat menulis laporan
                })

    df_results = pd.DataFrame(result_rows)
    df_timing = pd.DataFrame(timing_rows)

    df_results.to_csv("hasil_retrieval.csv", index=False)
    df_timing.to_csv("hasil_timing.csv", index=False)

    print(f"hasil_retrieval.csv: {df_results.shape[0]} baris disimpan")
    print(f"hasil_timing.csv: {df_timing.shape[0]} baris disimpan")


hasil_retrieval.csv: 290 baris disimpan
hasil_timing.csv: 30 baris disimpan


## 5. Analisis: Waktu Komputasi, Relevansi, dan Perbandingan Top-10

In [14]:
if len(documents) > 0:
    print("Rata-rata waktu komputasi per model (detik, dirata-rata atas 5 query):")
    timing_summary = df_timing.groupby("model")["time_sec"].agg(["mean", "std", "min", "max"]).sort_values("mean")
    print(timing_summary)

    print("\nWaktu komputasi per model per query (detik):")
    timing_pivot = df_timing.pivot(index="query", columns="model", values="time_sec")
    print(timing_pivot)


Rata-rata waktu komputasi per model (detik, dirata-rata atas 5 query):
                                mean       std       min       max
model                                                             
Word2Vec (TF-IDF weighted)  0.004562  0.002309  0.002878  0.008587
QL (No Smoothing)           0.008660  0.002938  0.005997  0.013611
TF-IDF                      0.013030  0.003032  0.010128  0.017115
QL (Laplace)                0.020782  0.005746  0.014804  0.030149
QL (Linear Interpolation)   0.024778  0.004889  0.016304  0.028967
TF                          0.062393  0.008399  0.056906  0.077096

Waktu komputasi per model per query (detik):
model                                           QL (Laplace)  \
query                                                          
bantuan sembako warga terdampak pandemi             0.030149   
covid 19 isolasi mandiri protokol kesehatan         0.020173   
olimpiade tokyo bulu tangkis indonesia medali       0.017917   
polisi korban tewas pembunu

In [15]:
import itertools

if len(documents) > 0:
    model_names = list(models.keys())

    for query_test in queries:
        print("=" * 80)
        print(f"Perbandingan Top-10 Doc ID untuk query: '{query_test}'")
        print("=" * 80)
        sub = df_results[df_results["query"] == query_test]
        pivot = sub.pivot_table(index="rank", columns="model", values="doc_id", aggfunc="first")
        pivot = pivot.reindex(columns=model_names)
        print(pivot.to_string())

        # Jaccard overlap of top-10 doc sets between every pair of models
        print("\nOverlap (Jaccard) top-10 antar model:")
        doc_sets = {m: set(sub[sub["model"] == m]["doc_id"]) for m in model_names}
        for m1, m2 in itertools.combinations(model_names, 2):
            s1, s2 = doc_sets[m1], doc_sets[m2]
            if s1 or s2:
                jaccard = len(s1 & s2) / len(s1 | s2)
            else:
                jaccard = 0.0
            print(f"  {m1} vs {m2}: {jaccard:.2f} ({len(s1 & s2)} dokumen sama)")
        print()


Perbandingan Top-10 Doc ID untuk query: 'ppkm darurat warga negara asing dilarang masuk'
model       TF   TF-IDF  Word2Vec (TF-IDF weighted)  QL (No Smoothing)  QL (Laplace)  QL (Linear Interpolation)
rank                                                                                                           
1       4350.0   4350.0                         0.0             3337.0        4350.0                     3337.0
2      11203.0   4374.0                      1236.0             4355.0        3337.0                     4355.0
3      11316.0   4380.0                      3740.0            10256.0        3370.0                    10256.0
4        706.0   2693.0                      4350.0             4350.0        3392.0                     4350.0
5       5229.0  11316.0                      4356.0                NaN        5229.0                        0.0
6       3026.0   2356.0                      4380.0                NaN        4458.0                     4356.0
7      12135.0 

In [16]:
if len(documents) > 0:
    print("Rata-rata term_overlap_ratio (proksi otomatis relevansi) per model:")
    relevance_summary = df_results.groupby("model")["term_overlap_ratio"].agg(["mean", "std"]).sort_values("mean", ascending=False)
    print(relevance_summary)

    print("\nCatatan: term_overlap_ratio hanya proksi otomatis (rasio kata query yang")
    print("muncul literal di dokumen). Ini TIDAK menggantikan evaluasi relevansi manual,")
    print("karena model semantik (Word2Vec) dan QL dgn smoothing bisa mengambil dokumen")
    print("relevan tanpa overlap kata literal yang tinggi. Kolom 'relevan_manual' di")
    print("hasil_retrieval.csv perlu diisi (Y/N) per baris saat menulis laporan, dengan")
    print("membaca content_snippet masing-masing dan menilai relevansinya terhadap query.")


Rata-rata term_overlap_ratio (proksi otomatis relevansi) per model:
                                mean       std
model                                         
QL (No Smoothing)           1.000000  0.000000
QL (Linear Interpolation)   0.930667  0.094399
QL (Laplace)                0.909095  0.113192
TF                          0.781476  0.189722
TF-IDF                      0.753619  0.203385
Word2Vec (TF-IDF weighted)  0.698143  0.219551

Catatan: term_overlap_ratio hanya proksi otomatis (rasio kata query yang
muncul literal di dokumen). Ini TIDAK menggantikan evaluasi relevansi manual,
karena model semantik (Word2Vec) dan QL dgn smoothing bisa mengambil dokumen
relevan tanpa overlap kata literal yang tinggi. Kolom 'relevan_manual' di
hasil_retrieval.csv perlu diisi (Y/N) per baris saat menulis laporan, dengan
membaca content_snippet masing-masing dan menilai relevansinya terhadap query.
